# Impact of the Gold Line Foothill Extension on LA Metro Ridership

**Research Question:** Did the March 2016 opening of the Gold Line Foothill Extension Phase 2A lead to a statistically significant change in ridership on the Gold Line compared to other Metro rail lines?

**Method:** Difference-in-differences (DiD) analysis comparing Gold Line (treatment) ridership trends against other Metro rail lines (control) before and after the extension opening.

**Data:** Monthly average weekday boardings by line, January 2009 â€“ June 2025, from the Streets For All ridership dashboard.

## Setup and Data Loading

In [ ]:
import sys
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src modules
project_root = pathlib.Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.viz import (
    set_default_theme,
    plot_ridership_trends,
    plot_treatment_vs_control,
    plot_distributions,
    plot_boxplot_comparison,
    plot_correlation_heatmap,
)
from src.stats import (
    summary_stats_table,
    shapiro_wilk_test,
    independent_ttest,
    mann_whitney_u,
)

set_default_theme()
print("Libraries loaded.")

Load the cleaned ridership and station location data produced by Phase 1. The ridership data has 944 rows covering 6 rail lines from 2009â€“2025, with treatment/control and pre/post period flags already assigned.

In [ ]:
ridership = pd.read_csv(project_root / "data" / "clean" / "ridership_clean.csv", parse_dates=["date"])
stations = pd.read_csv(project_root / "data" / "clean" / "station_locations_clean.csv", parse_dates=["opened_date"])

print(f"Ridership: {len(ridership)} rows, {ridership['date'].min().date()} to {ridership['date'].max().date()}")
print(f"Lines: {sorted(ridership['line_id'].unique())}")
print(f"Stations: {len(stations)} total, {stations['is_foothill_2016'].sum()} treatment")
ridership.head()

Define the primary analysis window. We focus on January 2012 â€“ December 2019 to have at least 4 years of pre-period data and to avoid COVID-era disruption. The Gold Line data begins in 2012 (line opened 2003, but Streets For All data for Gold starts 2012).

In [ ]:
# Primary analysis window: 2012-01 to 2019-12 (pre-COVID)
ANALYSIS_START = "2012-01-01"
ANALYSIS_END = "2019-12-31"
EXTENSION_DATE = pd.Timestamp("2016-03-05")

analysis_df = ridership[
    (ridership["date"] >= ANALYSIS_START) & (ridership["date"] <= ANALYSIS_END)
].copy()

# For primary DiD: gold = treatment, other established lines = control
# Exclude k_line (opened 2022) and a_line_merged (post-Regional Connector)
PRIMARY_LINES = ["gold", "a_line", "red", "green", "expo"]
analysis_df = analysis_df[analysis_df["line_id"].isin(PRIMARY_LINES)].copy()

# Create a simpler treatment label for plotting
analysis_df["group"] = analysis_df["line_id"].apply(
    lambda x: "Treatment (Gold)" if x == "gold" else "Control"
)

print(f"Analysis window: {ANALYSIS_START} to {ANALYSIS_END}")
print(f"Rows: {len(analysis_df)}")
print(f"Lines: {sorted(analysis_df['line_id'].unique())}")
print(f"Pre-extension rows: {(analysis_df['period'] == 'pre').sum()}")
print(f"Post-extension rows: {(analysis_df['period'] == 'post').sum()}")

---
## 1. Exploratory Data Analysis

### 1.1 Ridership Trends Over Time

The first visualization shows monthly average weekday boardings for each rail line from 2009 to 2019. The Gold Line (treatment) is highlighted. A vertical dashed line marks the Foothill Extension opening in March 2016.

We expect to see: (a) a general upward trend in system ridership leading up to 2016, (b) a visible bump or level shift in Gold Line ridership after the extension, and (c) relatively stable or declining trends in control lines for comparison.

In [ ]:
fig = plot_ridership_trends(
    ridership,
    line_ids=["gold", "a_line", "red", "green", "expo"],
    end_date="2019-12-31",
    title="Monthly Avg. Weekday Boardings by Line (2009â€“2019)",
)
plt.show()

### 1.2 Treatment vs Control â€” Aggregated View

To visualize the DiD design, we plot the Gold Line (treatment) against the average of all control lines. If the extension had a causal effect, we should see the Gold Line diverge from the control group trend after March 2016.

In [ ]:
fig = plot_treatment_vs_control(
    ridership,
    end_date="2019-12-31",
    title="Treatment (Gold Line) vs Control Lines â€” Avg. Weekday Boardings",
)
plt.show()

### 1.3 Full Timeline Including COVID and Recovery (2009â€“2025)

For context, here is the full data range including the COVID disruption (2020â€“2021) and recovery. The primary analysis excludes this period, but it informs our robustness checks later.

In [ ]:
fig = plot_ridership_trends(
    ridership,
    line_ids=["gold", "a_line", "red", "green", "expo", "a_line_merged"],
    end_date=None,
    title="Monthly Avg. Weekday Boardings â€” Full Timeline (2009â€“2025)",
)
plt.show()

### 1.4 Summary Statistics â€” Pre vs Post Extension

This table shows descriptive statistics (mean, median, std, IQR) for each line's weekday boardings, broken out by pre-extension (Jan 2012 â€“ Feb 2016) and post-extension (Apr 2016 â€“ Dec 2019) periods.

In [ ]:
# Summary stats by line and period
pre_df = analysis_df[analysis_df["period"] == "pre"]
post_df = analysis_df[analysis_df["period"] == "post"]

print("=== PRE-EXTENSION (Jan 2012 â€“ Feb 2016) ===")
pre_stats = summary_stats_table(pre_df, "line_id", "avg_daily_boardings")
display(pre_stats.round(0))

print("\n=== POST-EXTENSION (Apr 2016 â€“ Dec 2019) ===")
post_stats = summary_stats_table(post_df, "line_id", "avg_daily_boardings")
display(post_stats.round(0))

Compute the percentage change in mean ridership from pre to post for each line. This gives a quick view of which lines gained or lost riders after March 2016.

In [ ]:
# Percentage change in mean ridership: pre -> post
pct_change = pd.DataFrame({
    "Pre Mean": pre_stats["Mean"],
    "Post Mean": post_stats["Mean"],
})
pct_change["Change"] = pct_change["Post Mean"] - pct_change["Pre Mean"]
pct_change["% Change"] = (pct_change["Change"] / pct_change["Pre Mean"] * 100).round(1)

# Add treatment flag
pct_change["Group"] = pct_change.index.map(
    lambda x: "Treatment" if x == "gold" else "Control"
)
display(pct_change.sort_values("% Change", ascending=False))

### 1.5 Distribution of Ridership â€” Pre vs Post Extension

Histograms with KDE overlays show how the distribution of monthly ridership values changed after the extension. We split by treatment (Gold Line) and control groups for both pre and post periods. Comparing distribution shapes helps identify whether the extension shifted the entire distribution or just the mean.

In [ ]:
# Build groups for distribution plot: treatment pre/post, control pre/post
gold_pre = pre_df[pre_df["line_id"] == "gold"]["avg_daily_boardings"]
gold_post = post_df[post_df["line_id"] == "gold"]["avg_daily_boardings"]
control_pre = pre_df[pre_df["line_id"] != "gold"]["avg_daily_boardings"]
control_post = post_df[post_df["line_id"] != "gold"]["avg_daily_boardings"]

groups = {
    "Gold Line â€” Pre": gold_pre,
    "Gold Line â€” Post": gold_post,
    "Control Lines â€” Pre": control_pre,
    "Control Lines â€” Post": control_post,
}

fig = plot_distributions(
    ridership,
    groups=groups,
    metric_label="Avg. Daily Weekday Boardings",
    title="Ridership Distributions: Pre vs Post Extension",
)
plt.show()

### 1.6 Boxplot â€” Ridership by Line and Period

Boxplots provide a compact view of the median, interquartile range, and outliers for each line, split by pre/post period. This visualization makes it easy to spot whether the Gold Line's post-extension distribution shifted relative to control lines.

In [ ]:
# Create a combined label for boxplot: "line_name (period)"
box_df = analysis_df.copy()
box_df["line_period"] = box_df["line_id"].map(
    lambda x: "Gold Line" if x == "gold" else "Control"
) + " â€” " + box_df["period"].str.capitalize()

fig = plot_boxplot_comparison(
    box_df,
    x="line_period",
    y="avg_daily_boardings",
    title="Ridership Distribution by Group and Period",
    order=["Gold Line â€” Pre", "Gold Line â€” Post", "Control â€” Pre", "Control â€” Post"],
)
plt.show()

### 1.7 Normality Assessment â€” Shapiro-Wilk Tests

Before choosing between parametric (t-test) and non-parametric (Mann-Whitney U) tests in Phase 3, we check whether each group's ridership data is approximately normally distributed using the Shapiro-Wilk test. A p-value < 0.05 suggests the data departs from normality.

In [ ]:
# Run Shapiro-Wilk on each key group
normality_groups = {
    "Gold Line Pre": gold_pre,
    "Gold Line Post": gold_post,
    "Control Pre": control_pre,
    "Control Post": control_post,
}

normality_results = []
for name, values in normality_groups.items():
    result = shapiro_wilk_test(values, name=name)
    normality_results.append(result.to_dict())

normality_df = pd.DataFrame(normality_results)
display(normality_df)

# Plain-English summary
for name, values in normality_groups.items():
    result = shapiro_wilk_test(values, name=name)
    print(result.interpretation)

**Interpretation:** The Shapiro-Wilk results above determine our test strategy for Phase 3. Groups consistent with normality support parametric tests (t-tests); groups departing from normality will use non-parametric alternatives (Mann-Whitney U). We report both regardless, per our pre-registered analysis plan.

### 1.8 Line-to-Line Ridership Correlation

Since ACS demographic data was not incorporated (business proximity analysis dropped per D-01), we examine correlations between monthly ridership across lines. High positive correlations suggest shared system-wide trends (economy, fare changes, service quality). The Gold Line's correlation with control lines informs the parallel trends assumption needed for the DiD analysis in Phase 3.

In [ ]:
fig = plot_correlation_heatmap(
    analysis_df,
    line_ids=["gold", "a_line", "red", "green", "expo"],
    title="Line-to-Line Ridership Correlation (2012â€“2019, Monthly)",
)
plt.show()

**Interpretation:** High positive correlations between the Gold Line and control lines (especially in the pre-period) support the parallel trends assumption â€” these lines were moving together before the intervention. Any post-extension divergence is more plausibly attributable to the extension rather than unrelated factors.